# Round 6 (self-contained) — sibling search + expansion, vouch/twin features, entity decision
Inputs: **amc-2026** + **Notebook Output of `stage1_ALL`** only. CPU, Internet On. **Save Version → Save & Run All (Commit)**. ~7 h.
Outputs in `/kaggle/working/output/`: `matching_results.tsv` (auto-chosen rule), `matching_results_*.tsv` variants, `candidate_pairs.tsv`, `errors_*.tsv`, `test_scores.parquet`, `train_oof_scores.parquet`, models.

In [ ]:
!pip install -q --no-index --find-links=/kaggle/input/pip-wheels anyascii sparse_dot_topn rapidfuzz


In [ ]:
import os
data = [r for r, _, f in os.walk("/kaggle/input", followlinks=True)
        if "train_ground_truth.tsv" in f and r.endswith("/train") and os.path.exists(r[:-6] + "/test/test_source1.tsv")]
edges_dir = next((r for r, _, f in os.walk("/kaggle/input", followlinks=True)
                  if "edges_train_q1.parquet" in f and "norm_test.parquet" in f), None)
print("data:", data); print("stage-1 output:", edges_dir)
assert data, "competition data (amc-2026) not attached"
assert edges_dir, "stage1_ALL notebook output not attached"
os.environ["DATA_DIR"] = data[0][:-6]; os.environ["EDGES_DIR"] = edges_dir
os.makedirs("/kaggle/working/src/er", exist_ok=True)
open("/kaggle/working/src/er/__init__.py", "w").write("")
os.makedirs("/kaggle/working/er_work2", exist_ok=True)
for sp in ("train", "test"):     # reuse cleaned data from stage1_ALL (symlinks, no copy)
    dst = f"/kaggle/working/er_work2/norm_{sp}.parquet"
    if not os.path.lexists(dst):
        os.symlink(f"{edges_dir}/norm_{sp}.parquet", dst)
print(sorted(os.listdir(edges_dir)))

In [ ]:
%%writefile /kaggle/working/src/er/normalize.py
"""Text normalization for business names and addresses.

Every function is country-agnostic: no hard-coded state lists or country rules, so
France (unseen in training) goes through exactly the same path as US/India.
Licenses: anyascii (ISC), stdlib re.
"""
import re
from anyascii import anyascii

_WS = re.compile(r"\s+")
_DOTTED = re.compile(r"\b(?:[a-z]\.\s?){2,}")                      # "l.l.c." -> "llc"
_NONALNUM = re.compile(r"[^a-z0-9 ]")
_PHONE_TAIL = re.compile(r"\s+-\s+\+?\d[\d\s-]{6,}$")          # "... - 7149969359"
_PIPE_TAIL = re.compile(r"\s*\|.*$")                              # "... | www.x.com"
_URL = re.compile(r"^(?:https?://)?(?:www\.)?([a-z0-9\-]+)\.(?:com|net|org|in|co|biz|info|fr)$")
_ALIAS = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|dba|fka|f/k/a|formerly known as|doing business as|trading as|t/a)\b")
_LEADING_JUNK = re.compile(r"^[\W_]+")
_HONORIFIC = re.compile(r"^(?:m\s*/\s*s|messrs|shri|shree|sri|smt|mr|mrs|ms|the)\b\.?\s+")
_ORD = re.compile(r"\b(\d+)\s*(?:st|nd|rd|th)\b")
_LEAD0 = re.compile(r"\b0+(\d)")
_NULLTOK = re.compile(r"(?:^|(?<=[\s,]))(?:<?null>?|n/?a|none|nil|-+)(?=[\s,]|$)", re.I)
# leet: only inside tokens that also contain letters, so "15/58" or "2nd" stay numeric
_LEET = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "6": "g", "8": "b", "@": "a", "$": "s"})

def to_ascii(s: str) -> str:
    return anyascii(s or "").lower()

def _deleet_token(t: str) -> str:
    return t.translate(_LEET) if re.search(r"[a-z]", t) and re.search(r"\d", t) else t

def clean_name(s: str) -> str:
    """Canonical name string: ascii, lowercase, junk/phone/url/alias stripped, leet folded."""
    s = to_ascii(s).strip()
    s = _PHONE_TAIL.sub("", s)
    s = _PIPE_TAIL.sub("", s)
    s = _LEADING_JUNK.sub("", s)
    m = _URL.match(s.replace(" ", ""))
    if m:
        s = m.group(1).replace("-", " ")
    s = _DOTTED.sub(lambda m: m.group(0).replace(".", "").replace(" ", "") + " ", s)
    parts = _ALIAS.split(s)            # "X dba Y": keep both sides, separated
    s = " ".join(parts)
    s = _HONORIFIC.sub("", s)
    s = " ".join(_deleet_token(t) for t in s.split())
    s = s.replace("&", " and ").replace("+", " and ")
    s = _NONALNUM.sub(" ", s)
    toks = s.split()
    out = [t for i, t in enumerate(toks) if i == 0 or t != toks[i - 1]]   # "llc llc" -> "llc"
    return " ".join(out)

def clean_address(s: str) -> str:
    s = to_ascii(s)
    s = _NULLTOK.sub(" ", s)
    s = _DOTTED.sub(lambda m: m.group(0).replace(".", "").replace(" ", "") + " ", s)
    s = s.replace("#", " ")
    s = _ORD.sub(r"\1", s)
    s = _LEAD0.sub(r"\1", s)
    s = _NONALNUM.sub(" ", s)
    return _WS.sub(" ", s).strip()

_VOW = re.compile(r"[aeiouy]")
_SKEL_MAP = [("ph", "f"), ("w", "v"), ("q", "k"), ("c", "k"), ("z", "s"), ("x", "ks")]

def skeleton(s: str) -> str:
    """Consonant skeleton per token — bridges transliteration ('inovetiv pavr' ~ 'innovative power')."""
    out = []
    for t in s.split():
        if t.isdigit():
            continue
        for a, b in _SKEL_MAP:
            t = t.replace(a, b)
        t = t[0] + _VOW.sub("", t[1:]) if t else t
        t = re.sub(r"(.)\1+", r"\1", t)
        out.append(t)
    return " ".join(out)

def numbers(s: str) -> frozenset:
    return frozenset(re.findall(r"\d+", s))


In [ ]:
%%writefile /kaggle/working/src/er/data.py
"""Loading + cached normalization of the three sources."""
import csv, os, time
from multiprocessing import Pool
import numpy as np
import pandas as pd
from .normalize import clean_name, clean_address, skeleton

READ_KW = dict(sep="\t", dtype=str, keep_default_na=False, na_filter=False,
               quoting=csv.QUOTE_NONE, engine="c")

def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)

def read_tsv(path):
    return pd.read_csv(path, **READ_KW)

def _norm_chunk(args):
    names, addrs = args
    nm = [clean_name(x) for x in names]
    sk = [skeleton(x) for x in nm]
    ad = [clean_address(x) for x in addrs]
    return nm, sk, ad

def normalize_frame(df, n_proc=4, chunk=50_000):
    parts = [(df.business_name.values[i:i + chunk], df.business_address.values[i:i + chunk])
             for i in range(0, len(df), chunk)]
    with Pool(n_proc) as p:
        out = p.map(_norm_chunk, parts)
    df = df.copy()
    df["nm"] = [x for o in out for x in o[0]]
    df["sk"] = [x for o in out for x in o[1]]
    df["ad"] = [x for o in out for x in o[2]]
    df["na"] = df["nm"] + " " + df["ad"]
    return df

def load_split(data_dir, split, work_dir, n_proc=4):
    """Returns (s1, s23, gt_or_None). s23 = S2 and S3 concatenated. Normalized columns cached as parquet."""
    cache = os.path.join(work_dir, f"norm_{split}.parquet")
    if os.path.exists(cache):
        log(f"loading cached normalized data {cache}")
        allrec = pd.read_parquet(cache)
    else:
        frames = []
        for s in (1, 2, 3):
            p = os.path.join(data_dir, split, f"{split}_source{s}.tsv")
            t = time.time(); d = read_tsv(p); d["src"] = np.int8(s)
            log(f"read {p}: {len(d):,} rows ({time.time()-t:.0f}s)")
            frames.append(d)
        allrec = pd.concat(frames, ignore_index=True)
        t = time.time()
        allrec = normalize_frame(allrec, n_proc=n_proc)
        log(f"normalized {len(allrec):,} rows ({time.time()-t:.0f}s)")
        allrec.to_parquet(cache, index=False)
    s1 = allrec[allrec.src == 1].reset_index(drop=True)
    s23 = allrec[allrec.src != 1].reset_index(drop=True)
    gt = None
    gp = os.path.join(data_dir, split, f"{split}_ground_truth.tsv")
    if os.path.exists(gp):
        gt = read_tsv(gp)
    return s1, s23, gt

def truth_index(gt, s1, s23):
    """For each s23 row -> index of its true S1 row (or -1)."""
    s1_pos = pd.Series(np.arange(len(s1)), index=s1.entity_id.values)
    m = gt.matched_entity_ids.str.split(",")
    pairs = pd.DataFrame({"s1": gt.source1_entity_id.repeat(m.str.len()), "m": [x for l in m for x in l]})
    pairs = pairs[pairs.m != ""]
    owner = pd.Series(s1_pos.reindex(pairs.s1.values).values, index=pairs.m.values)
    return owner.reindex(s23.entity_id.values).fillna(-1).astype(np.int64).values


In [ ]:
%%writefile /kaggle/working/src/er/candidates.py
"""Multi-pass sparse TF-IDF candidate generation.

Direction: every S2/S3 record queries the S1 records of the same country and keeps its
top-k S1 neighbours per pass. (Each S2/S3 record belongs to at most one S1, so this is the
natural direction; per-S1 candidate lists are the inverse of these edges.)
Common tokens (document frequency > max_df) are dropped: little TF-IDF weight, most of the cost.
Identical query strings are searched once and the result is shared.
Licenses: scikit-learn (BSD-3), sparse_dot_topn (Apache-2.0), scipy/numpy (BSD).
"""
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from .data import log

# id, name, field, analyzer, ngram_range, max_df, k (neighbours kept)
PASSES = [
    (0, "na_w", "na", "word", (1, 1), 0.02, 5),
    (1, "ad_w", "ad", "word", (1, 2), 0.01, 5),
    (2, "nm_w", "nm", "word", (1, 1), 0.02, 1),
]
# Extra passes, run only on "hard" queries (non-Latin-script name, or empty/very short address):
# these are where the main passes miss most, and they are a small share of all records.
EXTRA_PASSES = [
    (3, "nm_w10", "nm", "word",    (1, 1), 0.02, 10),
    (4, "sk_c3",  "sk", "char_wb", (3, 3), 0.02, 5),
]
# Round-4 addition: skeleton + address words for hard queries (transliterated names with short addresses)
EXTRA2_PASSES = [
    (5, "sa_w", "sa", "word", (1, 1), 0.02, 10),
]
PASS_NAMES = {p[0]: p[1] for p in PASSES + EXTRA_PASSES + EXTRA2_PASSES}
PASS_NAMES[6] = "sib"      # candidates inherited from sibling S2/S3 records (round 6)

def sibling_query_mask(s23):
    """Records likely to be missed by the direct search: non-Latin name, short/empty address, 1-token name."""
    one_tok = (s23.nm.str.count(" ").values == 0)
    return hard_query_mask(s23) | one_tok

def sibling_search(s23, q_idx, n_threads, k=6, max_df=0.02, chunk=200_000):
    """Self-search: hard S2/S3 records vs ALL S2/S3 records of the same country on name+address words.
    Returns DataFrame(q, nb, sim) with nb != q."""
    out = []
    for c in sorted(s23.country.unique()):
        d_idx = np.flatnonzero(s23.country.values == c)
        b_idx = np.intersect1d(d_idx, q_idx, assume_unique=True)
        if len(b_idx) == 0:
            continue
        t = time.time()
        vec = TfidfVectorizer(analyzer="word", max_df=max_df, min_df=1, sublinear_tf=True, dtype=np.float32, lowercase=False)
        vec.fit(s23["na"].iloc[d_idx])
        D_T = vec.transform(s23["na"].iloc[d_idx]).T.tocsr()
        t_fit = time.time() - t; t = time.time()
        for i in range(0, len(b_idx), chunk):
            bi = b_idx[i:i + chunk]
            B = vec.transform(s23["na"].iloc[bi])
            C = sp_matmul_topn(B, D_T, top_n=k + 1, threshold=0.05, sort=True, n_threads=n_threads)
            rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
            q = bi[rows]; nb = d_idx[C.indices]; keep = q != nb
            out.append(pd.DataFrame({"q": q[keep].astype(np.int32), "nb": nb[keep].astype(np.int32), "sim": C.data[keep].astype(np.float32)}))
        log(f"  siblings country={c:8} docs={len(d_idx):>10,} queries={len(b_idx):>9,} fit {t_fit:4.0f}s  search {time.time()-t:5.0f}s")
        del D_T, vec
    S = pd.concat(out, ignore_index=True) if out else pd.DataFrame({"q": [], "nb": [], "sim": []})
    return S.sort_values(["q", "sim"], ascending=[True, False]).groupby("q").head(k).reset_index(drop=True)

def expand_via_siblings(S, E, top_rank=3):
    """Give each hard query the top-`top_rank` na_w candidates of its siblings (pass 6, score = sim * sibling score)."""
    nb_edges = E[(E["pass"] == 0) & (E["rank"] < top_rank)][["q", "s1", "score", "rank"]].rename(columns={"q": "nb"})
    X = S.merge(nb_edges, on="nb", how="inner")
    X["score"] = (X.sim * X.score).astype(np.float32)
    X = X.sort_values(["q", "score"], ascending=[True, False]).drop_duplicates(["q", "s1"])
    X["rank"] = X.groupby("q").cumcount().astype(np.int8)
    X["pass"] = np.int8(6)
    return X[["q", "s1", "score", "rank", "pass"]].reset_index(drop=True)

def hard_query_mask(s23):
    """Non-Latin-script name (transliterated) OR address with <= 3 tokens after cleaning."""
    raw = s23.business_name.values
    nonlatin = np.fromiter((any(ord(ch) > 0x2FF and ch.isalpha() for ch in x) for x in raw), bool, len(raw))
    short_addr = s23.ad.str.count(" ").values + (s23.ad.values != "") <= 3
    return nonlatin | short_addr

def run_pass(s1, s23, q_idx, spec, n_threads, countries=None, k_override=None, chunk=200_000):
    """Returns DataFrame(q, s1, score, rank, pass) with global row indices into s23 / s1."""
    pid, name, field, analyzer, ngram, max_df, k = spec
    k = k_override or k
    out = []
    cs = sorted(set(s1.country.unique()) & set(s23.country.unique()))
    if countries:
        cs = [c for c in cs if c in countries]
    for c in cs:
        a_idx = np.flatnonzero(s1.country.values == c)
        b_all = np.flatnonzero(s23.country.values == c)
        b_idx = np.intersect1d(b_all, q_idx, assume_unique=True)
        if len(a_idx) == 0 or len(b_idx) == 0:
            continue
        t = time.time()
        vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, max_df=max_df, min_df=1,
                              sublinear_tf=True, dtype=np.float32, lowercase=False)
        vec.fit(pd.concat([s1[field].iloc[a_idx], s23[field].iloc[b_all]], ignore_index=True))
        A_T = vec.transform(s1[field].iloc[a_idx]).T.tocsr()
        t_fit = time.time() - t; t = time.time()
        # search each distinct query string once
        codes, uniq = pd.factorize(s23[field].values[b_idx])
        res_q, res_s1, res_sc, res_rk = [], [], [], []
        for i in range(0, len(uniq), chunk):
            B = vec.transform(uniq[i:i + chunk])
            C = sp_matmul_topn(B, A_T, top_n=k, threshold=0.0, sort=True, n_threads=n_threads)
            rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
            res_q.append(rows + i); res_s1.append(C.indices); res_sc.append(C.data)
            res_rk.append(np.arange(C.nnz) - C.indptr[rows])
        uq = np.concatenate(res_q); us1 = np.concatenate(res_s1)
        usc = np.concatenate(res_sc); urk = np.concatenate(res_rk)
        # expand unique-string results back to every query row (vectorized join on string id)
        res = pd.DataFrame({"u": uq, "s1": a_idx[us1].astype(np.int32), "score": usc.astype(np.float32),
                            "rank": urk.astype(np.int8)})
        qdf = pd.DataFrame({"u": codes, "q": b_idx.astype(np.int32)})
        out.append(qdf.merge(res, on="u", how="inner")[["q", "s1", "score", "rank"]])
        del res, qdf
        log(f"  pass {name:5} country={c:8} S1={len(a_idx):>9,} queries={len(b_idx):>9,} "
            f"(unique {len(uniq):>9,}) vocab={len(vec.vocabulary_):>9,} fit {t_fit:5.0f}s  search {time.time()-t:5.0f}s")
        del A_T, vec
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        {"q": np.array([], np.int32), "s1": np.array([], np.int32), "score": np.array([], np.float32), "rank": np.array([], np.int8)})
    df["pass"] = np.int8(pid)
    return df

def union(edges, k=None):
    """Union of edges (optionally rank < k) -> unique (q, s1) pairs."""
    e = edges if k is None else edges[edges["rank"] < k]
    return e[["q", "s1"]].drop_duplicates()


In [ ]:
%%writefile /kaggle/working/src/er/features.py
"""Pair features for (S2/S3 query record, S1 candidate) pairs.

All string similarities are computed with rapidfuzz.process.cpdist (element-wise, multithreaded).
Licenses: rapidfuzz (MIT), numpy/pandas (BSD).
"""
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

N_PASS = 7   # 0 na_w, 1 ad_w, 2 nm_w@1, 3 nm_w@10 (hard), 4 sk_c3 (hard), 5 sa_w (hard, round 4), 6 sibling expansion (round 6)
OLD_PASS = {"na_w": 0, "ad_w": 1, "nm_w": 2, "sk_c3": 4}

def normalize_edges(E, hard=None):
    """Bring edges from either Stage-1 format into (q, s1, score, rank, pass:int) restricted to the
    production configuration (na@5, ad@5, nm@1, and for hard queries nm@10 + sk_c3@5)."""
    if not pd.api.types.is_integer_dtype(E["pass"]):   # old 5-pass measurement format (string names)
        parts = [E[(E["pass"] == "na_w") & (E["rank"] < 5)].assign(pass_id=0),
                 E[(E["pass"] == "ad_w") & (E["rank"] < 5)].assign(pass_id=1),
                 E[(E["pass"] == "nm_w") & (E["rank"] < 1)].assign(pass_id=2)]
        if hard is not None:
            h = hard[E.q.values]
            parts.append(E[(E["pass"] == "nm_w") & (E["rank"] < 10) & h].assign(pass_id=3))
            parts.append(E[(E["pass"] == "sk_c3") & (E["rank"] < 5) & h].assign(pass_id=4))
        E = pd.concat(parts, ignore_index=True).drop(columns="pass").rename(columns={"pass_id": "pass"})
    E = E[["q", "s1", "score", "rank", "pass"]].copy()
    E["pass"] = E["pass"].astype(np.int8)
    return E

def pivot_edges(E):
    """One row per (q, s1) with score/rank per pass (score 0, rank 99 when absent)."""
    E = E.sort_values(["q", "s1", "pass"])
    key = E.q.values.astype(np.int64) * 10_000_000 + E.s1.values
    uk, inv = np.unique(key, return_inverse=True)
    P = pd.DataFrame({"q": (uk // 10_000_000).astype(np.int32), "s1": (uk % 10_000_000).astype(np.int32)})
    for p in range(N_PASS):
        m = E["pass"].values == p
        sc = np.zeros(len(uk), np.float32); rk = np.full(len(uk), 99, np.int8)
        sc[inv[m]] = E.score.values[m]; rk[inv[m]] = E["rank"].values[m]
        P[f"sc{p}"] = sc; P[f"rk{p}"] = rk
    P["n_pass"] = sum((P[f"rk{p}"] < 99).astype(np.int8) for p in range(N_PASS))
    return P

_DIG = re.compile(r"\d+")

def _num_sets(arr):
    return [frozenset(_DIG.findall(x)) for x in arr]

def _num_pair_feats(qnums, snums):
    """Per pair: min |a-b| and min digit-edit distance over numbers NOT shared by both sides,
    and a 'decoy' flag: nothing shared but some pair of numbers is close (0 < |a-b| <= 30)."""
    from rapidfuzz.distance import Levenshtein
    n = len(qnums)
    mind = np.full(n, -1, np.float32); mined = np.full(n, -1, np.int8); close = np.zeros(n, np.int8)
    qun = np.zeros(n, np.int8); sun = np.zeros(n, np.int8)
    for i, (a, b) in enumerate(zip(qnums, snums)):
        if not a or not b:
            continue
        ua, ub = a - b, b - a
        qun[i] = min(len(ua), 127); sun[i] = min(len(ub), 127)
        if not ua or not ub:
            continue
        best = 1e9; be = 99
        for x in ua:
            xi = int(x[-9:])
            for y in ub:
                d = abs(xi - int(y[-9:]))
                if d < best: best = d
                e = Levenshtein.distance(x, y)
                if e < be: be = e
        mind[i] = min(best, 1e6); mined[i] = min(be, 99)
        if not (a & b) and 0 < best <= 30:
            close[i] = 1
    return mind, mined, close, qun, sun

def add_features(P, s1, s23, s1_name_freq, s23_name_freq, chunk=2_000_000, workers=-1):
    """Adds string/number/context features in place-chunks; returns new DataFrame."""
    out = []
    for i in range(0, len(P), chunk):
        c = P.iloc[i:i + chunk].copy()
        qi = c.q.values; si = c.s1.values
        qn, sn = s23.nm.values[qi], s1.nm.values[si]
        qa, sa = s23.ad.values[qi], s1.ad.values[si]
        qk, sk = s23.sk.values[qi], s1.sk.values[si]
        f = {}
        f["n_tset"] = process.cpdist(qn, sn, scorer=fuzz.token_set_ratio, workers=workers)
        f["n_tsort"] = process.cpdist(qn, sn, scorer=fuzz.token_sort_ratio, workers=workers)
        f["n_ratio"] = process.cpdist(qn, sn, scorer=fuzz.ratio, workers=workers)
        f["n_partial"] = process.cpdist(qn, sn, scorer=fuzz.partial_ratio, workers=workers)
        f["n_jw"] = process.cpdist(qn, sn, scorer=JaroWinkler.normalized_similarity, workers=workers)
        f["sk_tset"] = process.cpdist(qk, sk, scorer=fuzz.token_set_ratio, workers=workers)
        f["a_tset"] = process.cpdist(qa, sa, scorer=fuzz.token_set_ratio, workers=workers)
        f["a_ratio"] = process.cpdist(qa, sa, scorer=fuzz.ratio, workers=workers)
        f["a_partial"] = process.cpdist(qa, sa, scorer=fuzz.partial_ratio, workers=workers)
        f["a_tsort"] = process.cpdist(qa, sa, scorer=fuzz.token_sort_ratio, workers=workers)
        for k, v in f.items():
            c[k] = v.astype(np.float32)
        qlen_a = np.fromiter((len(x) for x in qa), np.int32, len(qa))
        c["q_addr_len"] = qlen_a
        c["q_addr_empty"] = (qlen_a == 0).astype(np.int8)
        c["n_len_q"] = np.fromiter((len(x) for x in qn), np.int16, len(qn))
        c["n_len_s"] = np.fromiter((len(x) for x in sn), np.int16, len(sn))
        # house / building numbers
        qnums, snums = _num_sets(qa), _num_sets(sa)
        inter = np.fromiter((len(a & b) for a, b in zip(qnums, snums)), np.int16, len(qa))
        qcnt = np.fromiter((len(a) for a in qnums), np.int16, len(qa))
        scnt = np.fromiter((len(b) for b in snums), np.int16, len(qa))
        c["num_shared"] = inter
        c["num_q"] = qcnt
        c["num_frac"] = np.where(qcnt > 0, inter / np.maximum(qcnt, 1), -1).astype(np.float32)
        c["num_conflict"] = ((qcnt > 0) & (scnt > 0) & (inter == 0)).astype(np.int8)
        mind, mined, close, qun, sun = _num_pair_feats(qnums, snums)
        c["num_min_diff"] = mind; c["num_min_edit"] = mined; c["num_close_only"] = close
        c["num_q_unmatched"] = qun; c["num_s_unmatched"] = sun
        # name genericity: how many S1 / S2+S3 records share this exact cleaned name
        c["s1_name_freq"] = s1_name_freq[si]
        c["q_name_freq"] = s23_name_freq[qi]
        c["q_src3"] = (s23.src.values[qi] == 3).astype(np.int8)
        c["q_nonlatin"] = s23.nonlatin.values[qi]
        out.append(c)
    return pd.concat(out, ignore_index=True)

def add_context(P):
    """Per-query context: where this candidate sits among the query's candidates."""
    g = P.groupby("q", sort=False)
    for col in ("sc0", "n_tset", "a_tset"):
        mx = g[col].transform("max")
        P[f"{col}_gap"] = (mx - P[col]).astype(np.float32)
        P[f"{col}_qrank"] = g[col].rank(ascending=False, method="min").astype(np.float32)
    comb = P.n_tset + P.a_tset
    P["comb"] = comb.astype(np.float32)
    P["comb_gap"] = (comb.groupby(P.q).transform("max") - comb).astype(np.float32)
    # margin between the query's best and second-best candidate (vectorized)
    o = np.lexsort((-comb.values, P.q.values))
    qs, cs = P.q.values[o], comb.values[o]
    first = np.r_[True, qs[1:] != qs[:-1]]
    pos = np.arange(len(qs)) - np.maximum.accumulate(np.where(first, np.arange(len(qs)), 0))
    top1 = pd.Series(cs[pos == 0], index=qs[pos == 0])
    top2 = pd.Series(cs[pos == 1], index=qs[pos == 1])
    t1 = top1.reindex(P.q.values).values; t2 = top2.reindex(P.q.values).fillna(0).values
    P["comb_margin_top"] = (t1 - t2).astype(np.float32)
    P["n_cands"] = g["s1"].transform("size").astype(np.int16)
    # S1-side: how many queries point at this S1 (density / chain signal)
    P["s1_indeg"] = P.groupby("s1")["q"].transform("size").astype(np.int32)
    return P

FEATURES = None  # filled by feature_columns()

def feature_columns(P):
    drop = {"q", "s1", "y"}
    return [c for c in P.columns if c not in drop]


In [ ]:
%%writefile /kaggle/working/src/er/pairsim.py
"""Exact TF-IDF similarities for EVERY candidate pair (not just the pass that retrieved it).

For each field a TF-IDF model is fitted on the split's own S1 + S2/S3 text (no labels involved).
Per pair we compute
  <field>_cos : cosine similarity of the two records' TF-IDF vectors
  <field>_max : largest single shared-term contribution (high = they share a RARE word / n-gram)
Fields: name words, address words (single digits kept, so house numbers count), name skeleton
character 3-grams (transliteration-robust).
Licenses: scikit-learn (BSD-3), scipy (BSD).
"""
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

FIELDS = [
    ("nmw", "nm", dict(analyzer="word", token_pattern=r"(?u)\b\w+\b")),
    ("adw", "ad", dict(analyzer="word", token_pattern=r"(?u)\b\w+\b")),
    ("skc", "sk", dict(analyzer="char_wb", ngram_range=(3, 3))),
]
SIM_COLS = [f"{n}_{k}" for n, _, _ in FIELDS for k in ("cos", "max")] + \
           [f"{n}_{k}" for n in ("nmw", "adw") for k in ("qonly", "sonly")]

class PairSim:
    def __init__(self, s1, s23, log=print):
        self.A, self.B = {}, {}
        for name, col, kw in FIELDS:
            vec = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, min_df=2, lowercase=False, **kw)
            vec.fit(pd.concat([s1[col], s23[col]], ignore_index=True))
            self.A[name] = vec.transform(s1[col]).tocsr()
            self.B[name] = vec.transform(s23[col]).tocsr()
            log(f"  pairsim {name}: vocab {len(vec.vocabulary_):,}  nnz S1 {self.A[name].nnz:,}  S2/S3 {self.B[name].nnz:,}")
            del vec

    def features(self, qi, si, block=1_000_000):
        out = {c: np.empty(len(qi), np.float32) for c in SIM_COLS}
        for i in range(0, len(qi), block):
            q, s = qi[i:i + block], si[i:i + block]
            for name, _, _ in FIELDS:
                Q = self.B[name][q]; S = self.A[name][s]
                P = Q.multiply(S).tocsr()
                out[f"{name}_cos"][i:i + block] = np.asarray(P.sum(axis=1)).ravel()
                out[f"{name}_max"][i:i + block] = P.max(axis=1).toarray().ravel()
                if name in ("nmw", "adw"):
                    # heaviest (rarest) term present on only ONE side: e.g. Public vs Private, 647 vs 636
                    q_only = (Q - Q.multiply(S.sign())).tocsr(); s_only = (S - S.multiply(Q.sign())).tocsr()
                    out[f"{name}_qonly"][i:i + block] = q_only.max(axis=1).toarray().ravel()
                    out[f"{name}_sonly"][i:i + block] = s_only.max(axis=1).toarray().ravel()
        return out


In [ ]:
%%writefile /kaggle/working/src/er/wordchange.py
"""Learned evidence from WHICH words differ between the two names.

Decoys and true matches both add / drop words, but different ones: noise adds 'center', 'services',
drops 'private limited'; decoys swap business words ('trading'->'mercantiles') or legal forms
('private'->'public'). For every word we learn, from labelled training pairs with similar names,
log-odds( true match | word present on only one side ). Tables are built out-of-fold: rows of fold f
are scored with a table learned on the other fold, so no label leaks into its own features.
"""
import numpy as np
import pandas as pd
from collections import Counter

WC_COLS = ["wc_q_min", "wc_q_sum", "wc_s_min", "wc_s_sum", "wc_n_q", "wc_n_s", "wc_unseen"]

def _diff(qn, sn):
    a, b = set(qn.split()), set(sn.split())
    return a - b, b - a, len(a & b) / max(len(a | b), 1)

def build_table(qn, sn, y, min_jacc=0.34):
    """Returns (dict word->log-odds for q-only words, dict for s-only words)."""
    cq = [Counter(), Counter()]; cs = [Counter(), Counter()]; tot = [0, 0]
    for a, b, lab in zip(qn, sn, y):
        qo, so, j = _diff(a, b)
        if j < min_jacc:
            continue
        tot[lab] += 1
        cq[lab].update(qo); cs[lab].update(so)
    base = np.log((tot[1] + 1) / (tot[0] + 1))
    def lo(c):
        words = set(c[0]) | set(c[1])
        return {w: float(np.log((c[1][w] + 1) / (c[0][w] + 1)) - base) for w in words if c[0][w] + c[1][w] >= 3}
    return lo(cq), lo(cs)

class WordChange:
    def __init__(self):
        self.tables = {}

    def fit(self, key, qn, sn, y):
        self.tables[key] = build_table(qn, sn, y)

    def features(self, qn, sn, keys):
        n = len(qn)
        out = {c: np.zeros(n, np.float32) for c in WC_COLS}
        for i, (a, b, k) in enumerate(zip(qn, sn, keys)):
            tq, ts = self.tables[k]
            qo, so, _ = _diff(a, b)
            unseen = 0
            if qo:
                v = [tq.get(w) for w in qo]; unseen += sum(x is None for x in v); v = [x for x in v if x is not None]
                if v:
                    out["wc_q_min"][i] = min(v); out["wc_q_sum"][i] = sum(v)
            if so:
                v = [ts.get(w) for w in so]; unseen += sum(x is None for x in v); v = [x for x in v if x is not None]
                if v:
                    out["wc_s_min"][i] = min(v); out["wc_s_sum"][i] = sum(v)
            out["wc_n_q"][i] = len(qo); out["wc_n_s"][i] = len(so); out["wc_unseen"][i] = unseen
        return out


In [ ]:
%%writefile /kaggle/working/src/er/decide.py
"""Decision rule + metric.

Each S2/S3 record belongs to at most one S1 (verified on the full training ground truth), so every
query is assigned to its single highest-probability candidate if that probability >= threshold.
"""
import numpy as np
import pandas as pd

def best_per_query(q, s1, p):
    o = np.lexsort((-p, q))
    q, s1, p = q[o], s1[o], p[o]
    first = np.r_[True, q[1:] != q[:-1]]
    return q[first], s1[first], p[first]

def f05(tp, fp, fn):
    """Per-entity F0.5 with the competition's singleton convention."""
    tp, fp, fn = np.asarray(tp, float), np.asarray(fp, float), np.asarray(fn, float)
    out = np.zeros_like(tp)
    single = (tp + fn) == 0
    out[single] = (fp[single] == 0).astype(float)
    ns = ~single
    denom = 1.25 * tp[ns] + 0.25 * fn[ns] + fp[ns]
    out[ns] = np.where(denom > 0, 1.25 * tp[ns] / np.maximum(denom, 1e-9), 0.0)
    return out

def entity_f05(assign_q, assign_s1, owner, n_s1, k_true):
    """Exact macro F0.5 over ALL S1 entities given assignments for ALL queries."""
    correct = owner[assign_q] == assign_s1
    tp = np.bincount(assign_s1[correct], minlength=n_s1)
    fp = np.bincount(assign_s1[~correct], minlength=n_s1)
    fn = k_true - tp
    return f05(tp, fp, fn).mean()

def simulated_entity_f05(val_q, val_best_s1, val_best_p, owner, k_true, n_queries_full, thresholds, rng_seed=0):
    """Macro F0.5 estimate when only a random SAMPLE of queries was scored.
    Per threshold: r = P(owned query correctly assigned), f = wrong assignments per query.
    Each entity with k true records gets TP ~ Bin(k, r), FP ~ Poisson(f * N_queries / N_S1)."""
    rng = np.random.default_rng(rng_seed)
    own = owner[val_q]
    res = []
    for t in thresholds:
        a = val_best_p >= t
        correct = a & (val_best_s1 == own)
        wrong = a & (val_best_s1 != own)
        r = correct[own >= 0].mean()
        f = wrong.mean()
        lam = f * n_queries_full / len(k_true)
        tp = rng.binomial(k_true, r)
        fp = rng.poisson(lam, size=len(k_true))
        res.append((t, f05(tp, fp, k_true - tp).mean(), r, f))
    return pd.DataFrame(res, columns=["threshold", "macro_f05_est", "recall_per_record", "wrong_per_query"])


# ---------------------------------------------------------------- round 6: per-entity expected-F0.5 decision
def entity_decide(bq, bs, bp, n_s1, K=8, M=48, seed=0, odds_ratio=1.0, chunk=200_000):
    """bq/bs/bp: each query's best S1 and probability. For every S1, records are added in order of
    probability while the Monte-Carlo expected macro-F0.5 of that entity increases (records are
    independent Bernoulli(p)). odds_ratio rescales p for a different match prior (test density).
    Returns boolean mask over (bq, bs, bp): assign or not."""
    p = bp.astype(np.float64)
    if odds_ratio != 1.0:
        o = p / np.maximum(1 - p, 1e-9) * odds_ratio; p = o / (1 + o)
    order = np.lexsort((-p, bs))
    s_sorted = bs[order]; p_sorted = p[order]
    first = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    start = np.flatnonzero(first); ends = np.r_[start[1:], len(s_sorted)]
    pos = np.arange(len(s_sorted)) - np.repeat(start, ends - start)
    n_ent = len(start)
    Pm = np.zeros((n_ent, K), np.float64)
    ent_id = np.repeat(np.arange(n_ent), ends - start)
    keep = pos < K
    Pm[ent_id[keep], pos[keep]] = p_sorted[keep]
    rng = np.random.default_rng(seed)
    a_star = np.zeros(n_ent, np.int64)
    for i in range(0, n_ent, chunk):
        Pc = Pm[i:i + chunk]                                   # (n, K)
        X = rng.random((M, *Pc.shape)) < Pc                    # (M, n, K)
        tp = np.concatenate([np.zeros((M, Pc.shape[0], 1)), np.cumsum(X, axis=2)], axis=2)   # (M, n, K+1)
        T = tp[..., -1:]
        a = np.arange(K + 1)[None, None, :]
        f = 1.25 * tp / np.maximum(1.25 * tp + 0.25 * (T - tp) + (a - tp), 1e-9)
        f[..., 0] = (T[..., 0] == 0)                           # empty prediction scores 1 iff no true record
        a_star[i:i + chunk] = f.mean(axis=0).argmax(axis=1)
    assign_sorted = pos < a_star[ent_id]
    out = np.zeros(len(bq), bool); out[order] = assign_sorted
    return out


In [ ]:
%%writefile /kaggle/working/src/er/context.py
"""Second-stage (entity-context) features built from first-stage probabilities of ALL pairs.

For a pair (q, s1) with first-stage probability p:
  query side : is p the query's best? gap to the query's best, the query's second-best p
  S1 side    : how strongly OTHER records point at the same S1 (max / sum / count of confident ones),
               split by whether they come from the same source (S2 vs S3) or the other one.
These capture the S2<->S3 cross-evidence and chain-store crowding without any transitive merging.
"""
import numpy as np
import pandas as pd

def _group_top2(keys, vals):
    """Per-row: max and second max of vals within its key group (keys need not be sorted)."""
    o = np.lexsort((-vals, keys))
    k, v = keys[o], vals[o]
    first = np.r_[True, k[1:] != k[:-1]]
    start = np.maximum.accumulate(np.where(first, np.arange(len(k)), 0))
    pos = np.arange(len(k)) - start
    ends = np.r_[np.flatnonzero(first)[1:], len(k)]
    grp_id = np.cumsum(first) - 1
    top1 = v[np.flatnonzero(first)]
    has2 = (ends - np.flatnonzero(first)) > 1
    top2 = np.where(has2, v[np.minimum(np.flatnonzero(first) + 1, len(v) - 1)], 0.0)
    t1 = np.empty(len(k), vals.dtype); t2 = np.empty(len(k), vals.dtype); p0 = np.empty(len(k), np.int32)
    t1[o] = top1[grp_id]; t2[o] = top2[grp_id]; p0[o] = pos
    return t1, t2, p0

def _group_sum(keys, vals, n):
    return np.bincount(keys, weights=vals, minlength=n)

def context_features(C, n_s1):
    """C: DataFrame with q, s1, p1, q_src3 (+ any base columns). Returns new feature columns in C."""
    q = C.q.values.astype(np.int64); s = C.s1.values.astype(np.int64)
    p = C.p1.values.astype(np.float32)
    # query side
    qt1, qt2, qpos = _group_top2(q, p)
    C["q_is_best"] = (qpos == 0).astype(np.int8)
    C["q_gap_best"] = (qt1 - p).astype(np.float32)
    C["q_second"] = np.where(qpos == 0, qt2, qt1).astype(np.float32)   # best competing candidate
    # S1 side, excluding the row itself
    st1, st2, spos = _group_top2(s, p)
    C["s1_max_other"] = np.where(spos == 0, st2, st1).astype(np.float32)
    ssum = _group_sum(s, p, n_s1)
    C["s1_sum_other"] = (ssum[s] - p).astype(np.float32)
    conf = ((qpos == 0) & (p >= 0.5)).astype(np.float64)       # records confidently assigned here
    scnt = _group_sum(s, conf, n_s1)
    C["s1_conf_other"] = (scnt[s] - conf).astype(np.float32)
    src = C.q_src3.values.astype(np.int64)
    key2 = s * 2 + src
    same_sum = _group_sum(key2, conf, n_s1 * 2)
    C["s1_conf_same_src"] = (same_sum[key2] - conf).astype(np.float32)
    C["s1_conf_other_src"] = (C["s1_conf_other"].values - C["s1_conf_same_src"].values).astype(np.float32)
    deg = np.bincount(s, minlength=n_s1)
    C["s1_deg"] = deg[s].astype(np.int32)
    return C

CTX_COLS = ["p1", "q_is_best", "q_gap_best", "q_second", "s1_max_other", "s1_sum_other", "s1_conf_other",
            "s1_conf_same_src", "s1_conf_other_src", "s1_deg"]


# ---------------------------------------------------------------- round 6 additions
SIB_COLS = ["sib_best_sim", "sib_vouch", "sib_vouch_sim", "sib_top_agree", "sib_n"]
TWIN_COLS = ["twin_num_other", "twin_name_other", "s1_conf_other_agreeaddr"]

def sibling_features(C, S):
    """C: pairs with q, s1, p1. S: sibling links (q, nb, sim). For each pair (q, X):
    sib_vouch = max over siblings r of sim(q,r) * p1(r, X); sib_top_agree = X is the best S1 of q's closest sibling."""
    n = len(C)
    for c in SIB_COLS:
        C[c] = np.zeros(n, np.float32)
    if S is None or len(S) == 0:
        return C
    P = C[["q", "s1", "p1"]]
    # sibling's best S1
    best = P.sort_values("p1", ascending=False).drop_duplicates("q").rename(columns={"q": "nb", "s1": "nb_best_s1", "p1": "nb_best_p"})
    S2 = S.merge(best, on="nb", how="left")
    # closest sibling per q
    top = S2.sort_values(["q", "sim"], ascending=[True, False]).drop_duplicates("q")[["q", "sim", "nb_best_s1"]]
    bs = C[["q", "s1"]].merge(top, on="q", how="left")
    C["sib_best_sim"] = bs.sim.fillna(0).values.astype(np.float32)
    C["sib_top_agree"] = (bs.nb_best_s1.values == bs.s1.values).astype(np.float32)
    C["sib_n"] = C.q.map(S.groupby("q").size()).fillna(0).values.astype(np.float32)
    # vouch: join sibling pairs (nb, X, p1) onto (q, X)
    nbp = P.loc[P.p1 >= 0.2].rename(columns={"q": "nb", "p1": "nb_p1"})   # only siblings' plausible pairs
    J = S[["q", "nb", "sim"]].merge(nbp, on="nb", how="inner")          # (q, nb, sim, s1, nb_p1)
    J["v"] = J.sim * J.nb_p1
    agg = J.groupby(["q", "s1"]).agg(v=("v", "max"), vs=("sim", "max")).reset_index()
    m = C[["q", "s1"]].merge(agg, on=["q", "s1"], how="left")
    C["sib_vouch"] = m.v.fillna(0).values.astype(np.float32)
    C["sib_vouch_sim"] = m.vs.fillna(0).values.astype(np.float32)
    return C

def twin_features(C, n_s1):
    """Evidence that S1 X has a near-duplicate business (a decoy twin) in the pool, from OTHER records:
    twin_num  = records whose name matches X well but whose house number is close-but-different,
    twin_name = records that share X's number but the model already rejects (name changed)."""
    s = C.s1.values.astype(np.int64)
    tn = ((C.num_close_only.values > 0) & (C.n_tset.values >= 80)).astype(np.float64)
    tw = ((C.num_frac.values >= 0.5) & (C.n_tset.values >= 55) & (C.p1.values < 0.5)).astype(np.float64)
    ag = ((C.num_frac.values >= 0.5) & (C.p1.values >= 0.5)).astype(np.float64)
    cn = np.bincount(s, weights=tn, minlength=n_s1); cw = np.bincount(s, weights=tw, minlength=n_s1); ca = np.bincount(s, weights=ag, minlength=n_s1)
    C["twin_num_other"] = (cn[s] - tn).astype(np.float32)
    C["twin_name_other"] = (cw[s] - tw).astype(np.float32)
    C["s1_conf_other_agreeaddr"] = (ca[s] - ag).astype(np.float32)
    return C


In [ ]:
%%writefile /kaggle/working/src/er/output.py
"""Submission files + official validator."""
import os, subprocess, sys
import numpy as np

def write_lists(path, header, q, s, s1_ids, s23_ids):
    """One row per S1 (all of them, in file order), comma-joined S2/S3 ids (may be empty)."""
    q = np.asarray(q); s = np.asarray(s)
    o = np.argsort(s, kind="stable"); q, s = q[o], s[o]
    idx = np.arange(len(s1_ids))
    starts = np.searchsorted(s, idx); ends = np.searchsorted(s, idx, side="right")
    with open(path, "w", encoding="utf-8") as f:
        f.write(header)
        for j in range(len(s1_ids)):
            ids = s23_ids[q[starts[j]:ends[j]]]
            f.write(s1_ids[j] + "\t" + ",".join(dict.fromkeys(ids)) + "\n")

def find_validator(data_dir):
    for root, _, files in os.walk("/kaggle/input", followlinks=True):
        if "validate_submission.py" in files:
            return os.path.join(root, "validate_submission.py")
    cand = os.path.join(os.path.dirname(data_dir.rstrip("/")), "utils", "validate_submission.py")
    return cand if os.path.exists(cand) else None

def run_validator(data_dir, mpath, cpath):
    val = find_validator(data_dir)
    if not val:
        print("validator not found"); return
    cmd = [sys.executable, val, "--matching", mpath, "--test-dir", os.path.join(data_dir, "test")]
    if cpath:
        cmd += ["--candidate", cpath]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])


In [ ]:
%%writefile /kaggle/working/src/run_stage1.py
#!/usr/bin/env python3
"""Stage 1 — candidate generation (+ recall report on train).

Full run:
    python run_stage1.py --data-dir DATASET_DIR --split test
    python run_stage1.py --data-dir DATASET_DIR --split train
Split across two notebooks by country (each writes its own edges file):
    ... --countries US
    ... --countries India France
Quick recall estimate on a random fraction of queries:
    ... --split train --query-frac 0.05
"""
import argparse, os, sys, time
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from er.data import load_split, truth_index, log
from er.candidates import PASSES, EXTRA_PASSES, EXTRA2_PASSES, PASS_NAMES, hard_query_mask, run_pass, union, \
    sibling_query_mask, sibling_search, expand_via_siblings

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True, help="folder containing train/ and test/")
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--work-dir", default="/kaggle/working/er_work")
    ap.add_argument("--query-frac", type=float, default=1.0)
    ap.add_argument("--countries", nargs="*", default=None, help="restrict to these country labels")
    ap.add_argument("--threads", type=int, default=os.cpu_count())
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--extra", action="store_true", help="run only the extra passes on hard queries")
    ap.add_argument("--extra2", action="store_true", help="run only the round-4 skeleton+address pass on hard queries")
    ap.add_argument("--siblings", action="store_true", help="round 6: self-search for hard records + candidate expansion via siblings")
    ap.add_argument("--main-edges", default=None, help="(--siblings) edges_<split>_q1.parquet of the main passes")
    a = ap.parse_args()
    os.makedirs(a.work_dir, exist_ok=True)
    T0 = time.time()

    s1, s23, gt = load_split(a.data_dir, a.split, a.work_dir, n_proc=a.threads)
    log(f"S1={len(s1):,}  S2+S3={len(s23):,}")
    log(f"countries S1={s1.country.value_counts().to_dict()}  S2+S3={s23.country.value_counts().to_dict()}")
    rng = np.random.default_rng(a.seed)
    q_idx = np.sort(rng.choice(len(s23), size=int(len(s23) * a.query_frac), replace=False)) if a.query_frac < 1 \
        else np.arange(len(s23))
    if a.countries:
        q_idx = q_idx[np.isin(s23.country.values[q_idx], a.countries)]
    passes = PASSES
    if a.siblings:
        s23["na"] = s23["nm"] + " " + s23["ad"]
        hard = sibling_query_mask(s23)
        q_idx = q_idx[hard[q_idx]]
        log(f"SIBLINGS mode: {hard.mean():.1%} of S2/S3 records are hard queries; self-search vs {len(s23):,} records")
        S = sibling_search(s23, q_idx, a.threads)
        sp = os.path.join(a.work_dir, f"edges_{a.split}_q1_sib.parquet"); S.to_parquet(sp, index=False)
        log(f"saved {sp} ({len(S):,} neighbour links, {S.q.nunique():,} queries)")
        E0 = pd.read_parquet(a.main_edges or os.path.join(a.work_dir, f"edges_{a.split}_q1.parquet"))
        X = expand_via_siblings(S, E0)
        xp = os.path.join(a.work_dir, f"edges_{a.split}_q1_extra3.parquet"); X.to_parquet(xp, index=False)
        log(f"saved {xp} ({len(X):,} expanded candidate pairs)")
        if gt is not None:
            owner = truth_index(gt, s1, s23)
            u0 = union(E0); f0 = set(u0.q.values[owner[u0.q.values] == u0.s1.values])
            hit = (owner[X.q.values] == X.s1.values)
            new = set(X.q.values[hit]) - f0
            n_true = int((owner >= 0).sum())
            print(f"\n  sibling expansion: {len(X):,} pairs; true S1 newly found for {len(new):,} records "
                  f"(+{len(new)/n_true:.2%} recall on top of main passes)")
        return
    if a.extra2:
        for d in (s1, s23):
            d["sa"] = d["sk"] + " " + d["ad"]
        hard = hard_query_mask(s23)
        q_idx = q_idx[hard[q_idx]]
        passes = EXTRA2_PASSES
        log(f"EXTRA2 mode: {hard.mean():.1%} of S2/S3 records are hard queries")
    elif a.extra:
        hard = hard_query_mask(s23)
        q_idx = q_idx[hard[q_idx]]
        passes = EXTRA_PASSES
        log(f"EXTRA mode: {hard.mean():.1%} of S2/S3 records are hard queries")
    log(f"querying {len(q_idx):,} S2/S3 records, countries={a.countries or 'all'}, threads={a.threads}")

    edges, times = [], {}
    for spec in passes:
        t = time.time()
        edges.append(run_pass(s1, s23, q_idx, spec, a.threads, countries=a.countries))
        times[spec[1]] = time.time() - t
    E = pd.concat(edges, ignore_index=True)
    tag = f"{a.split}_q{a.query_frac:g}" + ("_extra2" if a.extra2 else "_extra" if a.extra else "") + (f"_{'-'.join(a.countries)}" if a.countries else "")
    path = os.path.join(a.work_dir, f"edges_{tag}.parquet")
    E.to_parquet(path, index=False)
    log(f"saved {path} ({len(E):,} rows)")

    print("\n================ STAGE 1 REPORT ================")
    print(f"split={a.split} query_frac={a.query_frac} countries={a.countries or 'all'} total_time={(time.time()-T0)/60:.1f} min")
    for p, s in times.items():
        print(f"  pass {p:5}: {s/60:6.1f} min")
    u = union(E)
    print(f"  unique candidate pairs: {len(u):,}  ({len(u)/max(len(q_idx),1):.1f} per S2/S3 record)")
    n_s1_cands = u.s1.nunique()
    print(f"  S1 entities with >=1 candidate: {n_s1_cands:,} of {len(s1):,}")
    if gt is not None:
        owner = truth_index(gt, s1, s23)
        n_true = int((owner[q_idx] >= 0).sum())
        hit = (owner[u.q.values] == u.s1.values).sum()
        print(f"  recall (true S1 among candidates): {hit/n_true:.4f}  ({hit:,} of {n_true:,})")
        for pid in sorted(E["pass"].unique()):
            name = PASS_NAMES[int(pid)]; e = E[E["pass"] == pid]
            print(f"    {name:5}: {(owner[e.q.values] == e.s1.values).sum()/n_true:.4f}")
    print("================================================")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/run_round2.py
#!/usr/bin/env python3
"""Round 2 — full-data pipeline with out-of-fold validation and a second-stage context model.

  1. Pair features for ALL train candidate pairs (streamed in query chunks).
  2. Stage-A LightGBM, 2 folds split by S1 entity -> out-of-fold probability p1 for every train pair.
  3. Context features from p1 (query best/gap, S1-side support from other records, S2<->S3 cross-evidence).
  4. Stage-B LightGBM on base + context features, same folds -> out-of-fold p2.
  5. EXACT macro F0.5 over all 22 lakh train S1 entities for p1 and p2; best threshold chosen.
  6. Test: same features, fold models averaged, write matching_results.tsv + candidate_pairs.tsv, validate.

    python run_round2.py --data-dir DATASET_DIR --edges-dir STAGE1_OUTPUT/er_work --out-dir /kaggle/working/output
"""
import argparse, gc, os, shutil, sys, time
import numpy as np
import pandas as pd
import lightgbm as lgb
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from er.data import load_split, truth_index, log
from er.features import normalize_edges, pivot_edges, add_features, add_context
from er.context import context_features, CTX_COLS
from er.decide import best_per_query, entity_f05
from er.output import write_lists, run_validator

KEEP_BASE = ["sc0", "rk0", "comb", "comb_gap", "n_tset", "a_tset", "num_frac", "num_conflict",
             "q_addr_empty", "q_nonlatin", "s1_name_freq", "q_src3", "n_pass"]

def prep_tables(s1, s23):
    s23 = s23.copy()
    s23["nonlatin"] = np.fromiter((any(ord(ch) > 0x2FF and ch.isalpha() for ch in x) for x in s23.business_name.values),
                                  bool, len(s23)).astype(np.int8)
    k1 = s1.country + "|" + s1.nm
    s1_freq = k1.map(k1.value_counts()).values.astype(np.int32)
    k23 = s23.country + "|" + s23.nm
    s23_freq = k23.map(k23.value_counts()).values.astype(np.int32)
    return s23, s1_freq, s23_freq

def load_pairs(edges_dir, split, extra_dirs=()):
    names = (f"edges_{split}_q1.parquet", f"edges_{split}_q1_extra.parquet", f"edges_{split}_q1_extra2.parquet", f"edges_{split}_q1_extra3.parquet")
    parts = [pd.read_parquet(os.path.join(d, f)) for d in (edges_dir, *extra_dirs) for f in names
             if os.path.exists(os.path.join(d, f))]
    log(f"loaded {len(parts)} edge files for {split}")
    E = normalize_edges(pd.concat(parts, ignore_index=True))
    P = pivot_edges(E)            # sorted by q
    del E; gc.collect()
    return P

def chunks(P, n_q):
    qv = P.q.values
    uq = np.unique(qv)
    cuts = np.searchsorted(qv, np.r_[uq[::n_q], uq[-1] + 1])
    for i in range(len(cuts) - 1):
        yield P.iloc[cuts[i]:cuts[i + 1]]

def featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg_global):
    F = add_features(Pc.copy(), s1, s23, s1_freq, s23_freq)
    F = add_context(F)
    F["s1_indeg"] = s1_deg_global[F.s1.values]
    return F

PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=255, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=os.cpu_count(), verbose=-1)

def fit_two_folds(X, y, fold, name, lr=None):
    """Two fold-models with the SAME number of rounds (the first one's early-stopping point), so their
    probabilities are on the same scale. Memory: each fold's slice is binned by LightGBM, then freed."""
    models = []
    params = dict(PARAMS, **({"learning_rate": lr} if lr else {}))
    n_rounds = None
    for f in (0, 1):
        tr = fold == f
        t = time.time()
        d_tr = lgb.Dataset(X[tr], y[tr], free_raw_data=True); d_tr.construct()
        if n_rounds is None:
            va = ~tr
            d_va = lgb.Dataset(X[va], y[va], reference=d_tr, free_raw_data=True); d_va.construct()
            m = lgb.train(params, d_tr, num_boost_round=3000, valid_sets=[d_va],
                          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(200)])
            n_rounds = max(m.best_iteration, 50)
            del d_va
        else:
            m = lgb.train(params, d_tr, num_boost_round=n_rounds)
        del d_tr; gc.collect()
        log(f"{name} fold-model trained on fold {f}: {n_rounds} rounds ({time.time()-t:.0f}s)")
        models.append(m)
    return models      # models[f] was trained on fold f -> predicts rows of the OTHER fold

def predict_chunked(model, X_fn, n, rows=None, chunk=5_000_000):
    """Predict without materialising the full feature matrix. X_fn(idx) -> float32 matrix."""
    idx_all = np.arange(n) if rows is None else np.flatnonzero(rows)
    out = np.empty(len(idx_all), np.float32)
    for i in range(0, len(idx_all), chunk):
        idx = idx_all[i:i + chunk]
        out[i:i + chunk] = model.predict(X_fn(idx))
    return idx_all, out

def sweep(q, s1, p, owner, n_s1, k_true, label):
    bq, bs, bp = best_per_query(q, s1, p)
    rows = []
    for th in np.round(np.arange(0.20, 0.96, 0.025), 3):
        m = bp >= th
        rows.append((th, entity_f05(bq[m], bs[m], owner, n_s1, k_true)))
    R = pd.DataFrame(rows, columns=["threshold", label])
    return R

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--edges-dir", required=True, help="Stage-1 er_work folder (edges + norm caches)")
    ap.add_argument("--work-dir", default="/kaggle/working/r2")
    ap.add_argument("--out-dir", default="/kaggle/working/output")
    ap.add_argument("--train-sample", type=float, default=0.10, help="fraction of train queries used to fit models")
    ap.add_argument("--chunk", type=int, default=1_000_000, help="queries per feature chunk")
    ap.add_argument("--threshold", type=float, default=None)
    a = ap.parse_args()
    os.makedirs(a.work_dir, exist_ok=True); os.makedirs(a.out_dir, exist_ok=True)
    for sp in ("train", "test"):      # reuse Stage-1 normalization caches
        src = os.path.join(a.edges_dir, f"norm_{sp}.parquet"); dst = os.path.join(a.work_dir, f"norm_{sp}.parquet")
        if os.path.lexists(dst) and not os.path.exists(dst):
            os.remove(dst)            # broken link from an earlier run
        if os.path.exists(src) and not os.path.exists(dst):
            os.symlink(os.path.abspath(src), dst)
    T0 = time.time()

    # ============================================================ TRAIN
    s1, s23, gt = load_split(a.data_dir, "train", a.work_dir)
    s23, s1_freq, s23_freq = prep_tables(s1, s23)
    owner = truth_index(gt, s1, s23)
    k_true = np.bincount(owner[owner >= 0], minlength=len(s1))
    P = load_pairs(a.edges_dir, "train")
    log(f"train pairs: {len(P):,}  queries: {P.q.nunique():,}")
    s1_deg = np.bincount(P.s1.values, minlength=len(s1)).astype(np.int32)
    grp = np.where(owner >= 0, owner, 10_000_000 + np.arange(len(owner)))            # per query
    fold_q = (pd.util.hash_array(grp.astype(np.int64)) % 2).astype(np.int8)
    samp_q = (pd.util.hash_array(np.arange(len(owner), dtype=np.int64) + 7) % 10_000) < a.train_sample * 10_000

    # stream 1: features for the training sample
    parts = []
    for i, Pc in enumerate(chunks(P, a.chunk)):
        Pc = Pc[samp_q[Pc.q.values]]
        if len(Pc):
            parts.append(featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg))
        log(f"  stream1 chunk {i+1}: sample rows so far {sum(len(x) for x in parts):,}")
    T = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    feats = [c for c in T.columns if c not in ("q", "s1")]
    yT = (owner[T.q.values] == T.s1.values).astype(np.int8)
    fT = fold_q[T.q.values]
    log(f"stage-A training rows: {len(T):,}  positives {yT.mean():.2%}  features {len(feats)}")
    mA = fit_two_folds(T[feats].values.astype(np.float32), yT, fT, "stage-A")
    del T; gc.collect()

    # stream 2: out-of-fold p1 for every train pair (+ a few base columns for stage B)
    small = []
    for i, Pc in enumerate(chunks(P, a.chunk)):
        F = featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg)
        X = F[feats].values.astype(np.float32)
        fq = fold_q[F.q.values]
        p = np.empty(len(F), np.float32)
        p[fq == 1] = mA[0].predict(X[fq == 1], num_iteration=None)   # trained on fold 0
        p[fq == 0] = mA[1].predict(X[fq == 0], num_iteration=None)
        S = F[["q", "s1"] + KEEP_BASE].copy(); S["p1"] = p
        small.append(S)
        log(f"  stream2 chunk {i+1}: {len(F):,} pairs")
        del F, X
    C = pd.concat(small, ignore_index=True); del small, P; gc.collect()

    # stage B  (free the big text tables first)
    n_s1 = len(s1); s1_country = s1.country.values.copy()
    del s23; gc.collect()
    C = context_features(C, n_s1)
    featsB = KEEP_BASE + CTX_COLS
    yC = (owner[C.q.values] == C.s1.values).astype(np.int8)
    fC = fold_q[C.q.values]
    sC = samp_q[C.q.values]
    XB_s = C.loc[sC, featsB].values.astype(np.float32)
    mB = fit_two_folds(XB_s, yC[sC], fC[sC], "stage-B", lr=0.05)
    del XB_s; gc.collect()
    X_fn = lambda idx: C.iloc[idx][featsB].values.astype(np.float32)
    p2 = np.empty(len(C), np.float32)
    i1, v1 = predict_chunked(mB[0], X_fn, len(C), rows=(fC == 1)); p2[i1] = v1   # trained on fold 0
    i0, v0 = predict_chunked(mB[1], X_fn, len(C), rows=(fC == 0)); p2[i0] = v0
    gc.collect()

    # exact validation over ALL train S1 entities
    q, s = C.q.values, C.s1.values
    R1 = sweep(q, s, C.p1.values, owner, n_s1, k_true, "F05_stageA")
    R2 = sweep(q, s, p2, owner, n_s1, k_true, "F05_stageB")
    R = R1.merge(R2, on="threshold")
    bestA = R.loc[R.F05_stageA.idxmax()]; bestB = R.loc[R.F05_stageB.idxmax()]
    use_B = bestB.F05_stageB >= bestA.F05_stageA
    th = a.threshold if a.threshold is not None else float(bestB.threshold if use_B else bestA.threshold)
    cand_recall = (owner[q] == s).sum() / (owner >= 0).sum()
    print("\n================ ROUND 2 VALIDATION (exact, all train S1) ================")
    print(f"S1 entities: {n_s1:,}   candidate recall: {cand_recall:.4f}")
    print(R.iloc[::2].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"stage A best: {bestA.F05_stageA:.4f} @ {bestA.threshold}")
    print(f"stage B best: {bestB.F05_stageB:.4f} @ {bestB.threshold}")
    print(f"USING stage {'B' if use_B else 'A'} with threshold {th}")
    # breakdown for the chosen setting
    pb = p2 if use_B else C.p1.values
    bq, bs, bp = best_per_query(q, s, pb); m = bp >= th
    corr = owner[bq[m]] == bs[m]
    tp = np.bincount(bs[m][corr], minlength=n_s1); fp = np.bincount(bs[m][~corr], minlength=n_s1)
    from er.decide import f05
    ent = f05(tp, fp, k_true - tp)
    single = k_true == 0
    print(f"singletons: {single.mean():.2%} of S1, score {ent[single].mean():.4f} | non-singletons score {ent[~single].mean():.4f}")
    for c in sorted(set(s1_country)):
        cm = s1_country == c
        print(f"  {c:8}: macro F0.5 {ent[cm].mean():.4f}")
    imp = pd.Series(mB[0].feature_importance("gain"), index=featsB).sort_values(ascending=False)
    print("stage-B top features:", ", ".join(f"{k}:{v/imp.sum():.2f}" for k, v in imp.head(10).items()))
    print("==========================================================================")
    for i, m_ in enumerate(mA): m_.save_model(os.path.join(a.out_dir, f"stageA_fold{i}.txt"))
    for i, m_ in enumerate(mB): m_.save_model(os.path.join(a.out_dir, f"stageB_fold{i}.txt"))
    del C, p2, s1, gt, owner; gc.collect()

    # ============================================================ TEST
    s1t, s23t, _ = load_split(a.data_dir, "test", a.work_dir)
    s23t, s1t_freq, s23t_freq = prep_tables(s1t, s23t)
    Pt = load_pairs(a.edges_dir, "test")
    log(f"test pairs: {len(Pt):,}  S1 countries: {s1t.country.value_counts().to_dict()}")
    s1t_deg = np.bincount(Pt.s1.values, minlength=len(s1t)).astype(np.int32)
    small = []
    for i, Pc in enumerate(chunks(Pt, a.chunk)):
        F = featurize(Pc, s1t, s23t, s1t_freq, s23t_freq, s1t_deg)
        X = F[feats].values.astype(np.float32)
        p = 0.5 * (mA[0].predict(X, num_iteration=None) + mA[1].predict(X, num_iteration=None))
        S = F[["q", "s1"] + KEEP_BASE].copy(); S["p1"] = p.astype(np.float32)
        small.append(S)
        log(f"  test chunk {i+1}: {len(F):,} pairs")
        del F, X
    Ct = pd.concat(small, ignore_index=True); del small, Pt; gc.collect()
    if use_B:
        Ct = context_features(Ct, len(s1t))
        X_fn = lambda idx: Ct.iloc[idx][featsB].values.astype(np.float32)
        _, pa = predict_chunked(mB[0], X_fn, len(Ct)); _, pb_ = predict_chunked(mB[1], X_fn, len(Ct))
        pt = 0.5 * (pa + pb_)
    else:
        pt = Ct.p1.values
    bq, bs, bp = best_per_query(Ct.q.values, Ct.s1.values, pt)
    m = bp >= th
    s1_ids = s1t.entity_id.values; s23_ids = s23t.entity_id.values
    mpath = os.path.join(a.out_dir, "matching_results.tsv"); cpath = os.path.join(a.out_dir, "candidate_pairs.tsv")
    write_lists(mpath, "source1_entity_id\tmatched_entity_ids\n", bq[m], bs[m], s1_ids, s23_ids)
    write_lists(cpath, "source1_entity_id\tcandidate_entity_ids\n", Ct.q.values, Ct.s1.values, s1_ids, s23_ids)
    n_match = np.bincount(bs[m], minlength=len(s1_ids))
    print("\n================ ROUND 2 TEST OUTPUT ================")
    print(f"threshold {th} (stage {'B' if use_B else 'A'})  assigned {m.sum():,} of {len(bq):,} candidate-bearing records")
    print(f"S1 rows: {len(s1_ids):,}  with >=1 match: {(n_match>0).mean():.1%}  mean matches: {n_match.mean():.2f}")
    for c in sorted(s1t.country.unique()):
        cm = s1t.country.values == c
        print(f"  {c:8}: S1={cm.sum():>9,}  with match {(n_match[cm]>0).mean():.1%}  mean matches {n_match[cm].mean():.2f}")
    print(f"candidate pairs: {len(Ct):,}")
    run_validator(a.data_dir, mpath, cpath)
    print(f"total time {(time.time()-T0)/60:.1f} min")
    print("=====================================================")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/run_round3.py
#!/usr/bin/env python3
"""Round 3 — round 2 + exact pairwise TF-IDF similarities for every pair, error dumps, and a
memory-safe train -> predict flow in one run.

    python run_round3.py --data-dir DATASET_DIR --edges-dir STAGE1/er_work --out-dir /kaggle/working/output
"""
import argparse, gc, os, sys, time
import numpy as np
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from er.data import load_split, truth_index, log
from er.features import add_features, add_context
from er.pairsim import PairSim, SIM_COLS
from er.wordchange import WordChange, WC_COLS
from er.context import context_features, CTX_COLS, sibling_features, twin_features, SIB_COLS, TWIN_COLS
from er.decide import best_per_query, entity_f05, f05, entity_decide
from er.output import write_lists, run_validator
from run_round2 import prep_tables, load_pairs, chunks, fit_two_folds, predict_chunked, sweep

KEEP_BASE = ["sc0", "sc6", "comb_gap", "n_tset", "a_tset", "num_frac", "num_close_only", "q_addr_empty",
             "q_nonlatin", "s1_name_freq", "n_name_near", "wc_q_min", "q_src3"]

def featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg, sim, wc=None, wc_keys=None, V=None):
    F = add_features(Pc.copy(), s1, s23, s1_freq, s23_freq)
    if V is not None:
        F = F.merge(V, on=["q", "s1"], how="left"); F["v"] = F.v.fillna(0).astype(np.float32)
        F = F.rename(columns={"v": "sib_vouch_sc"})
        F["sib_vouch_sc_gap"] = (F.groupby("q", sort=False).sib_vouch_sc.transform("max") - F.sib_vouch_sc).astype(np.float32)
    else:
        F["sib_vouch_sc"] = np.float32(0); F["sib_vouch_sc_gap"] = np.float32(0)
    for k, v in sim.features(F.q.values, F.s1.values).items():
        F[k] = v
    # learned word-change evidence (out-of-fold tables on train, full table on test)
    if wc is not None:
        keys = wc_keys(F.q.values)
        for k, v in wc.features(s23.nm.values[F.q.values], s1.nm.values[F.s1.values], keys).items():
            F[k] = v
    g = F.groupby("q", sort=False)
    for c in ("nmw_cos", "adw_cos", "skc_cos"):
        F[f"{c}_gap"] = (g[c].transform("max") - F[c]).astype(np.float32)
    # ambiguity: how many candidates of this query carry a near-identical name
    F["n_name_near"] = (F.nmw_cos >= 0.9).groupby(F.q).transform("sum").astype(np.float32)
    F = add_context(F)
    F["s1_indeg"] = s1_deg[F.s1.values]
    return F

def weighted_sweep(q, s, p, owner, n_s1, k_true, ratio, label):
    """Macro F0.5 where each wrongly-assigned UNMATCHED record counts `ratio` times (test has more of them)."""
    bq, bs, bp = best_per_query(q, s, p)
    own = owner[bq]
    rows = []
    for th in np.round(np.arange(0.20, 0.96, 0.025), 3):
        m = bp >= th
        corr = m & (own == bs); wr_own = m & (own >= 0) & (own != bs); wr_un = m & (own < 0)
        tp = np.bincount(bs[corr], minlength=n_s1).astype(float)
        fp = np.bincount(bs[wr_own], minlength=n_s1) + ratio * np.bincount(bs[wr_un], minlength=n_s1)
        rows.append((th, f05(tp, fp, k_true - tp).mean()))
    return pd.DataFrame(rows, columns=["threshold", label])

def density_ratio(a, owner, n_s1, n_s23):
    """(unmatched S2/S3 records per S1 in test) / (same in train); true matches per S1 assumed equal."""
    tp = os.path.join(a.edges_dir, "norm_test.parquet")
    src = pd.read_parquet(tp, columns=["src"]).src.values
    n1t, n23t = int((src == 1).sum()), int((src != 1).sum())
    k = (owner >= 0).sum() / n_s1
    r = max((n23t / n1t - k) / (n_s23 / n_s1 - k), 1.0)
    log(f"density: train {n_s23/n_s1:.2f} S2/S3 per S1, test {n23t/n1t:.2f}; true matches per S1 {k:.2f} -> unmatched ratio {r:.2f}")
    return r

def load_siblings(a, split):
    for d in [a.edges_dir, *a.extra_edges_dir]:
        f = os.path.join(d, f"edges_{split}_q1_sib.parquet")
        if os.path.exists(f):
            S = pd.read_parquet(f); log(f"sibling links for {split}: {len(S):,}"); return S
    log(f"no sibling links for {split}"); return None

def vouch_table(P, S):
    """(q, s1) -> max over siblings r of sim(q,r) * na_w score(r, s1), and best sibling similarity for q."""
    if S is None or len(S) == 0:
        return None
    nbp = P.loc[P.sc0 > 0, ["q", "s1", "sc0"]].rename(columns={"q": "nb"})
    J = S[["q", "nb", "sim"]].merge(nbp, on="nb", how="inner")
    J["v"] = (J.sim * J.sc0).astype(np.float32)
    V = J.groupby(["q", "s1"], sort=False).v.max().reset_index()
    V["q"] = V.q.astype(np.int32); V["s1"] = V.s1.astype(np.int32)
    return V

def stage_b_features(C, n_s1, S):
    C = context_features(C, n_s1)
    C = sibling_features(C, S)
    C = twin_features(C, n_s1)
    return C

def link_norm(a, split):
    src = os.path.join(a.edges_dir, f"norm_{split}.parquet"); dst = os.path.join(a.work_dir, f"norm_{split}.parquet")
    if os.path.lexists(dst) and not os.path.exists(dst):
        os.remove(dst)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(os.path.abspath(src), dst)

def train_phase(a):
    link_norm(a, "train")
    s1, s23, gt = load_split(a.data_dir, "train", a.work_dir)
    s23, s1_freq, s23_freq = prep_tables(s1, s23)
    owner = truth_index(gt, s1, s23)
    k_true = np.bincount(owner[owner >= 0], minlength=len(s1))
    n_s1 = len(s1)
    sim = PairSim(s1, s23, log=log)
    P = load_pairs(a.edges_dir, "train", a.extra_edges_dir)
    log(f"train pairs: {len(P):,}")
    S_tr = load_siblings(a, "train"); V = vouch_table(P, S_tr)
    log(f"sibling vouch table: {0 if V is None else len(V):,} (q, s1) entries")
    s1_deg = np.bincount(P.s1.values, minlength=n_s1).astype(np.int32)
    grp = np.where(owner >= 0, owner, 10_000_000 + np.arange(len(owner)))
    fold_q = (pd.util.hash_array(grp.astype(np.int64)) % 2).astype(np.int8)
    samp_q = (pd.util.hash_array(np.arange(len(owner), dtype=np.int64) + 7) % 10_000) < a.train_sample * 10_000

    wc = WordChange()
    ps = samp_q[P.q.values]
    tq, ts = P.q.values[ps], P.s1.values[ps]
    ty = (owner[tq] == ts).astype(np.int8); tf = fold_q[tq]
    qn_all, sn_all = s23.nm.values[tq], s1.nm.values[ts]
    wc.fit(0, qn_all[tf == 1], sn_all[tf == 1], ty[tf == 1])      # used for fold-0 rows
    wc.fit(1, qn_all[tf == 0], sn_all[tf == 0], ty[tf == 0])      # used for fold-1 rows
    wc.fit("all", qn_all, sn_all, ty)                             # used for test
    log(f"word-change tables: q-only words {len(wc.tables['all'][0]):,}, s-only words {len(wc.tables['all'][1]):,}")
    del tq, ts, ty, tf, qn_all, sn_all
    oof_keys = lambda qv: fold_q[qv]
    parts = []
    for i, Pc in enumerate(chunks(P, a.chunk)):
        Pc = Pc[samp_q[Pc.q.values]]
        if len(Pc):
            parts.append(featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg, sim, wc, oof_keys, V))
    T = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    feats = [c for c in T.columns if c not in ("q", "s1")]
    yT = (owner[T.q.values] == T.s1.values).astype(np.int8); fT = fold_q[T.q.values]
    log(f"stage-A rows {len(T):,}  features {len(feats)}")
    XT = np.empty((len(T), len(feats)), np.float32)
    for k, c in enumerate(feats):                       # column-wise copy: no second full-size temporary
        XT[:, k] = T[c].values
    del T; gc.collect()
    mA = fit_two_folds(XT, yT, fT, "stage-A")
    impA = pd.Series(mA[0].feature_importance("gain"), index=feats).sort_values(ascending=False)
    del XT, yT, fT; gc.collect()

    small = []
    for i, Pc in enumerate(chunks(P, a.chunk)):
        t = time.time()
        F = featurize(Pc, s1, s23, s1_freq, s23_freq, s1_deg, sim, wc, oof_keys, V)
        X = F[feats].values.astype(np.float32); fq = fold_q[F.q.values]
        p = np.empty(len(F), np.float32)
        p[fq == 1] = mA[0].predict(X[fq == 1]); p[fq == 0] = mA[1].predict(X[fq == 0])
        S = pd.DataFrame({"q": F.q.values.astype(np.int32), "s1": F.s1.values.astype(np.int32), "p1": p})
        for c in KEEP_BASE:
            S[c] = F[c].values.astype(np.float32)
        small.append(S); del F, X, Pc; gc.collect()
        log(f"  stream2 chunk {i+1}: {len(S):,} pairs ({time.time()-t:.0f}s)")
    del P, sim; gc.collect()
    s23 = s23[["business_name", "business_address", "country"]].copy(); gc.collect()   # keep only text for error dumps
    C = pd.concat(small, ignore_index=True); del small; gc.collect()

    C = stage_b_features(C, n_s1, S_tr)
    featsB = KEEP_BASE + CTX_COLS + SIB_COLS + TWIN_COLS
    yC = (owner[C.q.values] == C.s1.values).astype(np.int8); fC = fold_q[C.q.values]; sC = samp_q[C.q.values]
    mB = fit_two_folds(C.loc[sC, featsB].values.astype(np.float32), yC[sC], fC[sC], "stage-B", lr=0.05)
    X_fn = lambda idx: C.iloc[idx][featsB].values.astype(np.float32)
    p2 = np.empty(len(C), np.float32)
    i1, v1 = predict_chunked(mB[0], X_fn, len(C), rows=(fC == 1)); p2[i1] = v1
    i0, v0 = predict_chunked(mB[1], X_fn, len(C), rows=(fC == 0)); p2[i0] = v0

    q, s = C.q.values, C.s1.values
    R = sweep(q, s, C.p1.values, owner, n_s1, k_true, "F05_stageA").merge(sweep(q, s, p2, owner, n_s1, k_true, "F05_stageB"), on="threshold")
    bA = R.loc[R.F05_stageA.idxmax()]; bB = R.loc[R.F05_stageB.idxmax()]
    use_B = bB.F05_stageB >= bA.F05_stageA
    th_train = float(bB.threshold if use_B else bA.threshold)
    pb = p2 if use_B else C.p1.values
    pd.DataFrame({"q": q, "s1": s, "p1": C.p1.values, "p2": p2, "y": (owner[q] == s).astype(np.int8)}).to_parquet(
        os.path.join(a.out_dir, "train_oof_scores.parquet"), index=False)
    ratio = density_ratio(a, owner, n_s1, len(s23))
    RW = weighted_sweep(q, s, pb, owner, n_s1, k_true, ratio, "F05_test_density")
    R = R.merge(RW, on="threshold")
    th = float(RW.loc[RW.F05_test_density.idxmax(), "threshold"])
    print(f"\n================ {a.tag} VALIDATION (exact, all train S1) ================")
    print(f"S1 entities {n_s1:,}   candidate recall {(owner[q] == s).sum() / (owner >= 0).sum():.4f}")
    print(R.iloc[::2].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"stage A best {bA.F05_stageA:.4f} @ {bA.threshold}   stage B best {bB.F05_stageB:.4f} @ {bB.threshold}")
    print(f"stage {'B' if use_B else 'A'}: best threshold at train density {th_train}, at TEST density (x{ratio:.2f} unmatched) {th}")
    print(f"USING stage {'B' if use_B else 'A'} @ {th}   (expected test-density score {RW.F05_test_density.max():.4f})")
    # per-entity expected-F0.5 decision vs the global threshold, both at train density
    bq_, bs_, bp_ = best_per_query(q, s, pb)
    m_ent = entity_decide(bq_, bs_, bp_, n_s1)
    f_ent = entity_f05(bq_[m_ent], bs_[m_ent], owner, n_s1, k_true)
    f_thr = entity_f05(bq_[bp_ >= th_train], bs_[bp_ >= th_train], owner, n_s1, k_true)
    use_entity = f_ent > f_thr + 0.0003
    print(f"decision rule at train density: global threshold {f_thr:.4f} | per-entity expected-F0.5 {f_ent:.4f} -> "
          f"{'ENTITY rule' if use_entity else 'THRESHOLD rule'} for test")
    bq, bs, bp = best_per_query(q, s, pb); m = bp >= th
    corr = owner[bq[m]] == bs[m]
    tp = np.bincount(bs[m][corr], minlength=n_s1); fp = np.bincount(bs[m][~corr], minlength=n_s1)
    ent = f05(tp, fp, k_true - tp); single = k_true == 0
    print(f"singletons score {ent[single].mean():.4f} | non-singletons {ent[~single].mean():.4f}")
    for c in sorted(set(s1.country.values)):
        print(f"  {c:8}: {ent[s1.country.values == c].mean():.4f}")
    # error accounting (record level)
    own_b = owner[bq]
    in_cand = np.zeros(len(owner), bool); in_cand[q[owner[q] == s]] = True
    n_true = (owner >= 0).sum()
    fn_nocand = ((owner >= 0) & ~in_cand).sum()
    fn_lowp = (in_cand[bq] & (bs == own_b) & ~m).sum()
    fn_wrongbest = (in_cand[bq] & (bs != own_b) & (own_b >= 0)).sum()
    fp_unowned = (m & (own_b < 0)).sum(); fp_wrongs1 = (m & (own_b >= 0) & (bs != own_b)).sum()
    print(f"true records {n_true:,}: missed-by-search {fn_nocand/n_true:.2%} | best-is-right-but-below-threshold {fn_lowp/n_true:.2%} "
          f"| a different S1 ranked best {fn_wrongbest/n_true:.2%}")
    print(f"false assignments: unmatched records assigned {fp_unowned:,} | assigned to the wrong S1 {fp_wrongs1:,}")
    print("stage-A top features:", ", ".join(f"{k}:{v/impA.sum():.3f}" for k, v in impA.head(15).items()))
    print("==========================================================================")
    # error samples with text, for inspection
    rng = np.random.default_rng(0)
    def dump(mask, name, n=1500):
        idx = np.flatnonzero(mask)
        idx = rng.choice(idx, size=min(n, len(idx)), replace=False)
        qq = bq[idx]; ss = bs[idx]; oo = own_b[idx]
        D = pd.DataFrame({"p": bp[idx], "q_name": s23.business_name.values[qq], "q_addr": s23.business_address.values[qq],
                          "country": s23.country.values[qq],
                          "pred_name": s1.business_name.values[ss], "pred_addr": s1.business_address.values[ss],
                          "true_name": np.where(oo >= 0, s1.business_name.values[np.maximum(oo, 0)], ""),
                          "true_addr": np.where(oo >= 0, s1.business_address.values[np.maximum(oo, 0)], "")})
        D.to_csv(os.path.join(a.out_dir, f"errors_{name}.tsv"), sep="\t", index=False)
    dump(m & (own_b < 0), "fp_unmatched")
    dump(m & (own_b >= 0) & (bs != own_b), "fp_wrong_s1")
    dump(in_cand[bq] & (bs == own_b) & ~m, "fn_below_threshold")
    for i, mm in enumerate(mA): mm.save_model(os.path.join(a.out_dir, f"stageA_fold{i}.txt"))
    for i, mm in enumerate(mB): mm.save_model(os.path.join(a.out_dir, f"stageB_fold{i}.txt"))
    a.extra_thresholds = sorted(set(a.extra_thresholds + [th_train]))
    a.use_entity = bool(use_entity)
    a.train_pairs_per_s1 = len(C) / n_s1; a.train_pos_per_s1 = (owner >= 0).sum() / n_s1 * (owner[q] == s).sum() / (owner >= 0).sum()
    return mA, mB, feats, featsB, use_B, th, wc

def test_phase(a, mA, mB, feats, featsB, use_B, th, wc):
    link_norm(a, "test")
    s1t, s23t, _ = load_split(a.data_dir, "test", a.work_dir)
    s23t, f1, f23 = prep_tables(s1t, s23t)
    s1_ids = s1t.entity_id.values.copy(); s23_ids = s23t.entity_id.values.copy(); cty = s1t.country.values.copy(); n_s1 = len(s1t)
    sim = PairSim(s1t, s23t, log=log)
    Pt = load_pairs(a.edges_dir, "test", a.extra_edges_dir)
    deg = np.bincount(Pt.s1.values, minlength=n_s1).astype(np.int32)
    S_te = load_siblings(a, "test"); Vt = vouch_table(Pt, S_te)
    parts = []
    for i, Pc in enumerate(chunks(Pt, a.chunk)):
        t = time.time()
        F = featurize(Pc, s1t, s23t, f1, f23, deg, sim, wc, lambda qv: ["all"] * len(qv), Vt)
        X = F[feats].values.astype(np.float32)
        S = pd.DataFrame({"q": F.q.values.astype(np.int32), "s1": F.s1.values.astype(np.int32),
                          "p1": (0.5 * (mA[0].predict(X) + mA[1].predict(X))).astype(np.float32)})
        for c in KEEP_BASE:
            S[c] = F[c].values.astype(np.float32)
        parts.append(S); del F, X, Pc; gc.collect()
        log(f"  test chunk {i+1}: {len(S):,} pairs ({time.time()-t:.0f}s)")
    del Pt, sim, s1t, s23t, f1, f23; gc.collect()
    Ct = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    Ct = stage_b_features(Ct, n_s1, S_te)
    X_fn = lambda idx: Ct.iloc[idx][featsB].values.astype(np.float32)
    _, pa = predict_chunked(mB[0], X_fn, len(Ct)); _, pb_ = predict_chunked(mB[1], X_fn, len(Ct))
    p2 = (0.5 * (pa + pb_)).astype(np.float32)
    cq = Ct.q.values.copy(); cs = Ct.s1.values.copy(); p1 = Ct.p1.values.copy()
    del Ct; gc.collect()
    os.makedirs(os.path.join(a.work_dir), exist_ok=True)
    pd.DataFrame({"q": cq, "s1": cs, "p1": p1, "p2": p2}).to_parquet(os.path.join(a.out_dir, "test_scores.parquet"), index=False)
    pt = p2 if use_B else p1
    bq, bs, bp = best_per_query(cq, cs, pt)
    mpath = os.path.join(a.out_dir, "matching_results.tsv"); cpath = os.path.join(a.out_dir, "candidate_pairs.tsv")
    # prior shift: test has more non-matching candidate pairs per S1 than train
    neg_tr = a.train_pairs_per_s1 - a.train_pos_per_s1; neg_te = len(cq) / n_s1 - a.train_pos_per_s1
    odds_ratio = float(np.clip(neg_tr / max(neg_te, 1e-9), 0.3, 1.0))
    print(f"test pairs per S1 {len(cq)/n_s1:.2f} vs train {a.train_pairs_per_s1:.2f} -> odds ratio {odds_ratio:.2f}")
    variants = [(f"th{th_:.3f}", bp >= th_) for th_ in sorted(set([th] + a.extra_thresholds))]
    variants.append(("entity", entity_decide(bq, bs, bp, n_s1, odds_ratio=odds_ratio)))
    variants.append(("entity_noshift", entity_decide(bq, bs, bp, n_s1)))
    main_key = "entity" if a.use_entity else f"th{th:.3f}"
    for key, m in variants:
        path = mpath if key == main_key else os.path.join(a.out_dir, f"matching_results_{key}.tsv")
        write_lists(path, "source1_entity_id\tmatched_entity_ids\n", bq[m], bs[m], s1_ids, s23_ids)
        n_match = np.bincount(bs[m], minlength=n_s1)
        print(f"{key:14} -> {os.path.basename(path)}: assigned {m.sum():,}, S1 with match {(n_match>0).mean():.1%}, "
              + ", ".join(f"{c} {(n_match[cty == c] > 0).mean():.1%}" for c in sorted(set(cty))))
    write_lists(cpath, "source1_entity_id\tcandidate_entity_ids\n", cq, cs, s1_ids, s23_ids)
    print(f"\n================ {a.tag} TEST OUTPUT ================")
    print(f"stage {'B' if use_B else 'A'}; main file = {main_key}")
    run_validator(a.data_dir, mpath, cpath)
    print("=====================================================")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--edges-dir", required=True)
    ap.add_argument("--work-dir", default="/kaggle/working/r3")
    ap.add_argument("--out-dir", default="/kaggle/working/output")
    ap.add_argument("--train-sample", type=float, default=0.10)
    ap.add_argument("--chunk", type=int, default=500_000)
    ap.add_argument("--extra-thresholds", type=float, nargs="*", default=[])
    ap.add_argument("--extra-edges-dir", nargs="*", default=[], help="more folders with edges_<split>_q1_extra2.parquet")
    ap.add_argument("--tag", default="ROUND 3")
    a = ap.parse_args()
    os.makedirs(a.work_dir, exist_ok=True); os.makedirs(a.out_dir, exist_ok=True)
    T0 = time.time()
    res = train_phase(a)
    gc.collect()
    log(f"train phase done ({(time.time()-T0)/60:.0f} min)")
    test_phase(a, *res)
    log(f"total {(time.time()-T0)/60:.0f} min")

if __name__ == "__main__":
    main()


## 1. Extra search passes: skeleton+address (hard records), then sibling self-search + expansion — ~2 h

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split train --work-dir /kaggle/working/er_work2 --extra2

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split test --work-dir /kaggle/working/er_work2 --extra2

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split train --work-dir /kaggle/working/er_work2 --siblings --main-edges "$EDGES_DIR/edges_train_q1.parquet"

In [ ]:
!python /kaggle/working/src/run_stage1.py --data-dir "$DATA_DIR" --split test --work-dir /kaggle/working/er_work2 --siblings --main-edges "$EDGES_DIR/edges_test_q1.parquet"

## 2. Train + validate + predict — ~5 h

In [ ]:
!python /kaggle/working/src/run_round3.py --data-dir "$DATA_DIR" --edges-dir "$EDGES_DIR" --extra-edges-dir /kaggle/working/er_work2 \
    --work-dir /kaggle/working/r6 --out-dir /kaggle/working/output --train-sample 0.10 --tag "ROUND 6"

In [ ]:
!ls -la /kaggle/working/output /kaggle/working/er_work2
!rm -f /kaggle/working/r6/norm_*.parquet /kaggle/working/er_work2/norm_*.parquet   # symlinks only